In [1]:
!git clone https://github.com/FaresAymanSoliman/resume-classification-ranking

Cloning into 'resume-classification-ranking'...
remote: Enumerating objects: 14535, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 14535 (delta 7), reused 14 (delta 7), pack-reused 14520 (from 1)
Receiving objects: 100% (14535/14535), 212.72 MiB | 22.03 MiB/s, done.
Resolving deltas: 100% (1574/1574), done.


In [2]:
%cd resume-classification-ranking/src/resume_pipeline/

/content/resume-classification-ranking/src/resume_pipeline


In [3]:
!git checkout feature/resume-pipeline
!git fetch 

Updating files: 100% (12649/12649), done.
Branch 'feature/resume-pipeline' set up to track remote branch 'feature/resume-pipeline' from 'origin'.
Switched to a new branch 'feature/resume-pipeline'


In [4]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/resume-classification-ranking")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())
print("data exists:", (PROJECT_ROOT / "data").exists())

Project root: /content/resume-classification-ranking
src exists: True
data exists: True


In [5]:
from src.utils.file_loader import load_and_prepare_resume_data
from src.resume_pipeline.preprocessing import preprocess_resume_dataframe
from src.resume_pipeline.features import (
    fit_transform_resume_features,
    save_vectorizer,
)
from src.resume_pipeline.model import (
    encode_labels,
    split_data,
    train_classifier,
    evaluate_classifier,
    save_model,
)

In [6]:
df = load_and_prepare_resume_data()

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Dataset shape: (2483, 3)
Columns: ['resume_id', 'resume_text', 'category']


,resume_id,resume_text,category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADM...,HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS Summary ...",HR
2,33176873,HR DIRECTOR Summary Over 20 years e...,HR
3,27018550,"HR SPECIALIST Summary Dedicated, Driv...",HR
4,17812897,HR MANAGER Skill Highlights ...,HR


In [7]:
df["category"].value_counts().head(10)

,count
category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,119
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


In [8]:
processed_df = preprocess_resume_dataframe(df)

print("Processed shape:", processed_df.shape)
processed_df[["resume_text", "cleaned_resume_text"]].head(3)

Processed shape: (2483, 4)


,resume_text,cleaned_resume_text
0,HR ADMINISTRATOR/MARKETING ASSOCIATE\n\nHR ADM...,hr administrator marketing associate hr admini...
1,"HR SPECIALIST, US HR OPERATIONS Summary ...",hr specialist us hr operations summary versati...
2,HR DIRECTOR Summary Over 20 years e...,hr director summary over 20 years experience i...


In [9]:
vectorizer, X = fit_transform_resume_features(processed_df)

print("Feature matrix shape:", X.shape)

Feature matrix shape: (2483, 5000)


In [10]:
y_encoded, label_encoder = encode_labels(
    processed_df["category"].values
)

print("Number of classes:", len(label_encoder.classes_))
print("Classes:", label_encoder.classes_)

Number of classes: 24
Classes: ['ACCOUNTANT' 'ADVOCATE' 'AGRICULTURE' 'APPAREL' 'ARTS' 'AUTOMOBILE'
 'AVIATION' 'BANKING' 'BPO' 'BUSINESS-DEVELOPMENT' 'CHEF' 'CONSTRUCTION'
 'CONSULTANT' 'DESIGNER' 'DIGITAL-MEDIA' 'ENGINEERING' 'FINANCE' 'FITNESS'
 'HEALTHCARE' 'HR' 'INFORMATION-TECHNOLOGY' 'PUBLIC-RELATIONS' 'SALES'
 'TEACHER']


In [11]:
X_train, X_test, y_train, y_test = split_data(X, y_encoded)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (1986, 5000)
Test size: (497, 5000)


In [12]:
model = train_classifier(X_train, y_train)
print("Model trained successfully")

Model trained successfully


In [13]:
metrics = evaluate_classifier(
    model=model,
    X_test=X_test,
    y_test=y_test,
    label_encoder=label_encoder,
)

print("Accuracy:", metrics["accuracy"])

Accuracy: 0.6398390342052314


In [14]:
import pandas as pd

report_df = pd.DataFrame(
    metrics["classification_report"]
).transpose()

report_df

,precision,recall,f1-score,support
ACCOUNTANT,0.625000,0.833333,0.714286,24.000000
ADVOCATE,0.476190,0.416667,0.444444,24.000000
AGRICULTURE,0.600000,0.461538,0.521739,13.000000
APPAREL,0.444444,0.210526,0.285714,19.000000
ARTS,0.400000,0.190476,0.258065,21.000000
AUTOMOBILE,0.833333,0.714286,0.769231,7.000000
AVIATION,0.850000,0.708333,0.772727,24.000000
BANKING,0.833333,0.652174,0.731707,23.000000
BPO,0.000000,0.000000,0.000000,4.000000
BUSINESS-DEVELOPMENT,0.525000,0.875000,0.656250,24.000000


In [15]:
save_model(model, label_encoder)
save_vectorizer(vectorizer)

print("✅ Model and vectorizer saved successfully")

Model saved to /content/resume-classification-ranking/models/classifier.pkl
Vectorizer saved to: /content/resume-classification-ranking/models/vectorizer.pkl
✅ Model and vectorizer saved successfully
